In [0]:
%pip install scikit-learn xgboost
dbutils.library.restartPython()

# 04: Machine Learning Model Comparison

**Question:** Can more flexible models predict which acquired sellers will start selling, and do they find patterns the logistic regression missed?

**Target:** `is_active_90d` (made at least one sale in the first 90 days).

**Features (known at signing only, to avoid leakage):** acquisition channel, business segment, lead type, business type, sales rep, days from first contact to close, signing date.

**Models:** baseline, logistic regression, random forest, gradient boosting, XGBoost.

**Evaluation:** 5-fold cross-validation repeated 5 times (25 train/test splits), because a single split on 667 sellers would be unreliable.


In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

df = spark.table("workspace.marts.fact_seller_channel_performance").toPandas()
df["is_active"] = df["is_active_90d"].astype(int)
df["days_to_close"] = df["days_to_close"].astype(float)

for col in ["business_segment", "lead_type", "business_type", "sr_id"]:
    df[col] = df[col].fillna("unknown")

df["days_since_start"] = (pd.to_datetime(df["won_date"]) - pd.Timestamp("2017-12-01")).dt.days.astype(float)

categorical = ["channel_group", "business_segment", "lead_type", "business_type", "sr_id"]
numeric = ["days_to_close", "days_since_start"]

X = df[categorical + numeric]
y = df["is_active"]

print(f"Sellers: {len(X)}, features: {X.shape[1]}, activation rate: {y.mean():.3f}")

In [0]:
preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10, sparse_output=False), categorical),
    ("num", StandardScaler(), numeric),
])

models = {
    "Baseline (no features)": DummyClassifier(strategy="prior"),
    "Logistic regression": LogisticRegression(C=1.0, max_iter=2000),
    "Random forest": RandomForestClassifier(
        n_estimators=500, min_samples_leaf=10, max_features="sqrt", random_state=42),
    "Gradient boosting": HistGradientBoostingClassifier(
        max_depth=3, learning_rate=0.05, max_iter=200, min_samples_leaf=20, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        min_child_weight=5, reg_lambda=1.0, eval_metric="logloss", random_state=42),
}

pipelines = {name: Pipeline([("prep", preprocess), ("model", model)]) for name, model in models.items()}

In [0]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "brier": "neg_brier_score",
}

fold_scores = {}
rows = []
for name, pipe in pipelines.items():
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    fold_scores[name] = scores["test_roc_auc"]
    rows.append({
        "model": name,
        "roc_auc_mean": scores["test_roc_auc"].mean(),
        "roc_auc_sd": scores["test_roc_auc"].std(),
        "pr_auc_mean": scores["test_pr_auc"].mean(),
        "brier_score": -scores["test_brier"].mean(),
    })

results = pd.DataFrame(rows).set_index("model").round(3)
results

In [0]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(list(fold_scores.values()), tick_labels=list(fold_scores.keys()))
ax.axhline(0.5, linestyle="--", color="gray", label="Coin flip (0.5)")
ax.set_ylabel("ROC-AUC across 25 test folds")
ax.set_title("Model comparison: predicting 90-day activation")
ax.legend()
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

logit = fold_scores["Logistic regression"]
for name in ["Random forest", "Gradient boosting", "XGBoost"]:
    diff = fold_scores[name] - logit
    print(f"{name} vs logistic: mean AUC difference = {diff.mean():+.3f}, "
          f"better in {(diff > 0).sum()} of {len(diff)} folds")

In [0]:
complex_models = ["Random forest", "Gradient boosting", "XGBoost"]
best_complex = results.loc[complex_models, "roc_auc_mean"].idxmax()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

importance = {}
for name in ["Logistic regression", best_complex]:
    fitted = clone(pipelines[name]).fit(X_train, y_train)
    perm = permutation_importance(fitted, X_test, y_test, scoring="roc_auc", n_repeats=30, random_state=42)
    importance[name] = pd.Series(perm.importances_mean, index=X.columns)

importance_table = pd.DataFrame(importance).sort_values(best_complex, ascending=False).round(4)
print(f"Best complex model: {best_complex}\n")
importance_table

## Results summary

| Model | ROC-AUC (mean ± SD, 25 folds) | PR-AUC | Brier score |
|---|---|---|---|
| Baseline (no features) | 0.500 | 0.433 | 0.246 |
| Logistic regression | 0.644 ± 0.041 | 0.580 | 0.236 |
| Random forest | 0.646 ± 0.040 | 0.572 | 0.231 |
| Gradient boosting | 0.617 ± 0.038 | 0.544 | 0.245 |
| XGBoost | 0.623 ± 0.039 | 0.555 | 0.246 |

1. **More complex models did not improve prediction.** Random forest tied logistic regression (+0.002 AUC, better in 14 of 25 folds). Gradient boosting and XGBoost performed worse (better in only 3 and 5 of 25 folds), a sign of overfitting on a small dataset. The relationships are simple enough for a linear model, so logistic regression is preferred for its interpretability.
2. **Signing-time information predicts activation only modestly** (AUC about 0.64). Most of what determines whether a seller starts selling is not captured in this data.
3. **Seller characteristics matter more than channel.** Business type, lead type, and business segment were the most important features in both models; acquisition channel ranked near the bottom.
4. **Significance vs. predictive power:** the regression showed that paid search has a real, significant effect on activation (+16.7 points, p = 0.001), but because it applies to only one channel and is small relative to seller-level variation, channel adds little predictive power overall.

**Business implication:** lead qualification based on seller characteristics (business type, lead type, segment) may improve activation more than shifting budget between channels.